---
# `Length Based Text Splitting`
---

### Introduction
- It is the fastest and Easiest Text Splitting 
- Here we split the data, where we already decides that how many characters or how many tokens we need to have in every chunks

Big disadvantage
- In a chunk, it might not even start or end as a proper word
- Length based text splitting, it can't even capture linguistic meaning, semantic meaning as sentence can break in between

Parameters
- chunk_size: maximum lenght of each chunk is character/tokens
- chunk_overlap: overlap between chunks to preserve content and it should be 10% or 20% of chunk_size

Document Loader + Text Splitter Actually Works?
- PDF ->

# `Detailed Notes`

# Length-Based Text Splitting in LangChain

> **Length-based text splitting divides a large text into smaller chunks based on a specified size, such as characters or tokens.**

It is one of the simplest approaches to text splitting and is useful when you need predictable chunk sizes.

---

# 1. Why Do We Need Length-Based Splitting?

Suppose we have a large document:

```text
Large Document
      ↓
50,000 characters
```

Sending the entire document to an LLM is not always practical.

Instead:

```text
50,000 characters
       ↓
Length-Based Splitter
       ↓
┌──────────────┐
│ Chunk 1      │
│ Chunk 2      │
│ Chunk 3      │
│ ...          │
│ Chunk 50     │
└──────────────┘
```

Each chunk has a controlled size.

---

# 2. Basic Idea

Suppose:

```text
Text length = 5,000 characters
```

and we configure:

```python
chunk_size = 1,000
```

Conceptually:

```text
5,000 characters
       ↓
┌──────────┐
│ Chunk 1  │ 1000
├──────────┤
│ Chunk 2  │ 1000
├──────────┤
│ Chunk 3  │ 1000
├──────────┤
│ Chunk 4  │ 1000
├──────────┤
│ Chunk 5  │ 1000
└──────────┘
```

So:

> **Length determines where the text is split.**

---

# 3. What Does "Length" Mean?

Length can mean different things:

### Character length

```text
1000 characters
```

### Token length

```text
500 tokens
```

### Word length

```text
200 words
```

The exact definition depends on the splitter being used.

---

# 4. Character-Based Length Splitting

A simple approach is:

```text
Split after approximately N characters.
```

For example:

```python
chunk_size = 500
```

The text is divided into chunks around 500 characters.

Conceptually:

```text
Text
 ↓
500 chars
 ↓
500 chars
 ↓
500 chars
 ↓
...
```

---

# 5. LangChain Example

One common implementation is:

```python
from langchain_text_splitters import CharacterTextSplitter
```

Example:

```python
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_text(text)
```

Here:

```text
chunk_size = 500
chunk_overlap = 50
```

means we want chunks around the configured size, with some shared content between neighboring chunks.

---

# 6. Understanding `chunk_size`

`chunk_size` controls the desired size of the chunks.

Example:

```python
chunk_size=1000
```

means:

> Try to create chunks around 1000 units of the splitter's length measure.

For a character-based splitter, this is approximately character length.

---

# 7. Understanding `chunk_overlap`

Suppose:

```python
chunk_size=1000
chunk_overlap=200
```

The next chunk shares approximately 200 units of content with the previous chunk.

Conceptually:

```text
Chunk 1
┌──────────────────────────────┐
│          1000 chars          │
└──────────────────────────────┘
                    ┌──────────────────────────────┐
                    │          1000 chars          │
                    └──────────────────────────────┘
                    ← 200 →
                      overlap
```

Why?

To avoid losing context at chunk boundaries.

---

# 8. Example of Overlap

Original:

```text
A B C D E F G H I J K L
```

Suppose:

```text
chunk_size = 6
chunk_overlap = 2
```

Conceptually:

```text
Chunk 1:
A B C D E F

Chunk 2:
E F G H I J

Chunk 3:
I J K L
```

Notice:

```text
Chunk 1 → E F
Chunk 2 → E F
```

and:

```text
Chunk 2 → I J
Chunk 3 → I J
```

The repeated content preserves continuity.

---

# 9. Why Is Overlap Important?

Imagine a sentence:

```text
The company's leave policy provides 20 days of annual
leave to employees who have completed one year of service.
```

If we split at an unfortunate position:

```text
Chunk 1:
The company's leave policy provides 20 days of annual

Chunk 2:
leave to employees who have completed one year of service.
```

The meaning is distributed across chunks.

Overlap can preserve some of the surrounding context.

---

# 10. Length-Based vs Recursive Splitting

This distinction is important.

## Character/Length-Based

Main goal:

> Control the size of chunks.

```text
Text
 ↓
Fixed approximate length
 ↓
Chunks
```

## Recursive Character Splitting

Main goal:

> Control chunk size while attempting to preserve natural text boundaries.

```text
Paragraph
 ↓
Sentence
 ↓
Word
 ↓
Character
```

### Simple Difference

```text
Length-based
→ "How much text should each chunk contain?"

Recursive
→ "How can I create appropriately sized chunks
   while preserving natural boundaries?"
```

---

# 11. Why Not Always Use Fixed-Length Splitting?

Consider:

```text
Machine learning is a field of artificial intelligence.
It enables computers to learn from data.

Supervised learning uses labeled data.
Classification and regression are common examples.
```

A fixed-length splitter might produce:

```text
Chunk 1:
Machine learning is a field of artificial intelligence.
It enables computers to learn from data.

Supervised lea
```

and:

```text
Chunk 2:
rning uses labeled data.
Classification and regression...
```

A sentence can be broken in the middle.

This may hurt readability and retrieval quality.

---

# 12. When Is Length-Based Splitting Useful?

Length-based splitting is useful when:

### 1. You need predictable chunk sizes

For example:

```text
Every ~500 characters
```

### 2. You have simple text

For example:

```text
Logs
Raw text
Simple strings
Generated text
```

### 3. You need basic chunking

You don't always need a complex semantic strategy.

### 4. You have strict size constraints

For example, when controlling token/context budgets.

---

# 13. Character Length vs Token Length

This is a very important GenAI concept.

LLMs don't fundamentally process text as characters.

They process **tokens**.

For example:

```text
"LangChain is useful"
```

is converted by a tokenizer into tokens.

Conceptually:

```text
Text
 ↓
Tokenizer
 ↓
Tokens
```

Therefore, token-based splitting can sometimes give more direct control over the amount of text sent to an LLM.

---

# 14. Token-Based Length Splitting

Instead of:

```text
chunk_size = 1000 characters
```

you might use:

```text
chunk_size = 500 tokens
```

Conceptually:

```text
Document
    ↓
Tokenizer
    ↓
Tokens
    ↓
500-token chunks
```

This is useful when your main concern is the model's context window.

---

# 15. Character Length ≠ Token Length

This is an important interview point.

Suppose:

```text
1000 characters
```

does **not** necessarily mean:

```text
1000 tokens
```

The relationship depends on:

* Language
* Vocabulary
* Tokenizer
* Text content
* Special characters
* Code

So:

> **Character-based chunk size and token-based chunk size are different measurements.**

---

# 16. Length-Based Splitting in RAG

Let's connect this to RAG.

```text
                DOCUMENT
                    ↓
             Length Splitter
                    ↓
                  Chunks
                    ↓
               Embeddings
                    ↓
              Vector Database
                    ↓
                Retriever
                    ↓
            Relevant Chunks
                    ↓
                   LLM
                    ↓
                 Answer
```

The splitter controls the size of the units that eventually become searchable.

---

# 17. Example: PDF RAG

Suppose:

```text
AI_book.pdf
```

contains:

```text
100 pages
```

Pipeline:

```text
PDF
 ↓
PDF Loader
 ↓
Documents
 ↓
Length-Based Splitter
 ↓
Chunks
 ↓
Embeddings
 ↓
Vector DB
```

Example:

```python
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter

loader = PyPDFLoader("AI_book.pdf")

documents = loader.load()

splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print(f"Documents: {len(documents)}")
print(f"Chunks: {len(chunks)}")
```

---

# 18. Important Issue: Separator

Character-based splitting can use a separator.

For example:

```text
"\n\n"
```

means paragraph boundaries.

Or:

```text
"\n"
```

means line boundaries.

Or:

```text
" "
```

means spaces.

Conceptually:

```text
Paragraph
   ↓
separator
   ↓
Next paragraph
```

This is one reason recursive splitting can be more flexible: it can try multiple separators.

---

# 19. Example

Suppose:

```text
Paragraph 1

Paragraph 2

Paragraph 3
```

Using:

```python
separator="\n\n"
```

the splitter can use paragraph boundaries.

So instead of cutting arbitrarily:

```text
Paragraph 1 + half of Paragraph 2
```

it can try to split between paragraphs.

---

# 20. Length-Based Splitting for Code

Suppose:

```python
def login():
    ...
```

and:

```python
def register():
    ...
```

A purely length-based splitter might cut:

```python
def login():
    validate_user()
    authenticate_user()
```

in the middle of a function.

For code, a language-aware splitter is usually preferable.

### Rule

> **Use length-based splitting when size control is more important than preserving complex structure.**

---

# 21. Choosing Chunk Size

There is no universal perfect value.

You might start with:

```python
chunk_size=500
chunk_overlap=50
```

or:

```python
chunk_size=1000
chunk_overlap=200
```

Then evaluate your RAG system.

### Don't say:

> "1000 is the best chunk size."

Instead:

> **"Chunk size is application-dependent. I start with a reasonable baseline and tune it based on document structure, retrieval quality, and model context requirements."**

That's a strong interview answer.

---

# 22. What If Chunk Size Is Too Small?

Example:

```text
chunk_size = 100
```

You may get:

```text
Chunk 1 → Definition
Chunk 2 → Explanation
Chunk 3 → Example
Chunk 4 → Conclusion
```

The chunks may lose context.

Potential problems:

* Poor semantic context
* Less meaningful retrieval
* More chunks
* More embeddings
* Higher storage cost

---

# 23. What If Chunk Size Is Too Large?

Example:

```text
chunk_size = 5000
```

You may get:

```text
Chunk 1
────────────────────────────
Many unrelated topics
Definitions
Examples
Tables
Conclusions
────────────────────────────
```

Potential problems:

* Retrieval becomes less precise
* More irrelevant context
* Higher token usage
* More information passed to the LLM

---

# 24. Chunking Is a Trade-Off

Remember:

```text
Small Chunks
    ↓
More precise
    +
Less context

Large Chunks
    ↓
More context
    +
More irrelevant information
```

The goal is:

> **Enough context + high retrieval precision**

---

# 25. Practical Project

## Project: Chat With a PDF

Suppose you have:

```text
deep_learning.pdf
```

### Step 1 — Load

```text
PDF
 ↓
PyPDFLoader
 ↓
Documents
```

### Step 2 — Length-Based Split

```text
Documents
 ↓
CharacterTextSplitter
 ↓
Chunks
```

### Step 3 — Embed

```text
Chunks
 ↓
Embedding Model
 ↓
Vectors
```

### Step 4 — Store

```text
Vectors
 ↓
Vector Database
```

### Step 5 — Retrieve

```text
Question
 ↓
Retriever
 ↓
Relevant Chunks
```

### Step 6 — Generate

```text
Question + Chunks
 ↓
LLM
 ↓
Answer
```

---

# 26. Complete Architecture

```text
                  PDF
                   │
                   ↓
              PDF Loader
                   │
                   ↓
               Documents
                   │
                   ↓
        Length-Based Text Splitter
                   │
                   ↓
                 Chunks
                   │
                   ↓
             Embedding Model
                   │
                   ↓
             Vector Database
                   │
        ───────────┼───────────
                   │
                   ↓
              User Query
                   │
                   ↓
               Retriever
                   │
                   ↓
            Relevant Chunks
                   │
                   ↓
                  LLM
                   │
                   ↓
                Answer
```

---

# 27. Important Interview Questions

## Beginner

### Q1. What is length-based text splitting?

**Answer:**

Length-based text splitting divides a document into chunks according to a predefined size, such as a number of characters or tokens.

---

### Q2. What is `chunk_size`?

**Answer:**

`chunk_size` specifies the desired maximum or approximate size of each chunk according to the splitter's measurement method.

---

### Q3. What is `chunk_overlap`?

**Answer:**

It specifies how much content should be shared between consecutive chunks to preserve context across chunk boundaries.

---

## Intermediate

### Q4. What is the difference between character-based and token-based splitting?

**Answer:**

Character-based splitting measures chunk size using characters, while token-based splitting measures it using tokens produced by a tokenizer. Token-based splitting provides more direct control over LLM context usage.

---

### Q5. Why can fixed-length splitting be problematic?

**Answer:**

It can split text in the middle of sentences, paragraphs, or logical units, which may reduce semantic coherence and retrieval quality.

---

### Q6. Why use overlap?

**Answer:**

Overlap helps preserve information that spans chunk boundaries, reducing the chance that important context is lost between consecutive chunks.

---

# 28. Scenario-Based Questions

### Q7. Your RAG system gives incomplete answers. What would you check?

I would check:

```text
Chunk size
Chunk overlap
Splitting strategy
Retrieved chunks
Embedding quality
Retriever configuration
```

The issue may be caused by chunks that are too small or by important context being separated across chunks.

---

### Q8. Your chunks contain too much unrelated information. What would you do?

I would consider reducing the chunk size or using a more structure-aware/semantic splitting strategy.

---

### Q9. Would you use fixed-length splitting for source code?

**Answer:**

Usually not as the first choice. Code-aware splitting is generally preferable because functions, classes, and logical blocks should ideally remain intact.

---

### Q10. How would you select the chunk size for a production RAG system?

**Answer:**

I'd consider document structure, query patterns, embedding model, LLM context limits, and retrieval quality. I'd establish a baseline and evaluate different configurations using real queries rather than assuming a universal value.

---

# 29. 30-Second Revision

> **Length-based splitting divides text according to a predefined length, usually characters or tokens.**

Remember:

```text
Large Text
   ↓
Length-Based Splitter
   ↓
Fixed/Controlled Size Chunks
```

### Important Parameters

```text
chunk_size
→ Size of each chunk

chunk_overlap
→ Shared content between chunks
```

### Main Trade-Off

```text
Small chunks
→ Precise but less context

Large chunks
→ More context but more noise
```

---

# 30. 2-Minute Revision

## Length-Based Text Splitting

Length-based splitting divides large text into smaller chunks based on a specified length.

### Character-Based

```python
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
```

### Token-Based

```text
Text
 ↓
Tokenizer
 ↓
Tokens
 ↓
Fixed token-sized chunks
```

### Why?

* Control chunk size
* Manage context
* Improve retrieval
* Reduce unnecessary tokens
* Prepare documents for embeddings

### Important Difference

```text
Character-based
→ Size measured in characters

Token-based
→ Size measured in tokens
```

### Compared with Recursive Splitting

```text
Length-Based
→ Focuses primarily on size

Recursive
→ Focuses on size + natural text boundaries
```

### RAG

```text
Loader
 ↓
Documents
 ↓
Length-Based Splitter
 ↓
Chunks
 ↓
Embeddings
 ↓
Vector DB
 ↓
Retriever
 ↓
LLM
```

### Interview One-Liner

> **Length-based text splitting divides documents into controlled-size chunks based on characters or tokens. `chunk_size` controls chunk length and `chunk_overlap` preserves context between chunks. However, the optimal size depends on the document type, retrieval task, and model, so it should be evaluated rather than treated as a fixed value.**
